# Data Analysis 

In [43]:
## Dependencies
import yfinance as yf
import pandas as pd
import numpy as np

### 1. Acquiring datasets

In [44]:
# NVIDIA dataset:
nvidia = yf.download("NVDA", start="2020-01-01", end="2026-07-01")
nvidia.columns = nvidia.columns.get_level_values(0)
nvidia.columns.name = None # removing title of column names.
nvidia.reset_index(inplace=True) # resetting index to have date as a column.

nvidia["Date"] = pd.to_datetime(nvidia["Date"]) # date column as Date.
nvidia.head(5)

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,2020-01-02,5.963805,5.963805,5.884506,5.934969,237536000
1,2020-01-03,5.868347,5.912098,5.819375,5.844234,205384000
2,2020-01-06,5.892957,5.898177,5.749026,5.775128,262636000
3,2020-01-07,5.964302,6.010041,5.876302,5.921296,314856000
4,2020-01-08,5.975488,6.016753,5.920053,5.960075,277108000


In [45]:
# Semiconductor billing dataset:
sia = pd.read_csv('americas_semiconductor_billings.csv') # one value per month; low granularity compared to nvidia dataset.
sia["Date"] = pd.to_datetime(sia["Date"])
sia.head(5)

/var/folders/7n/53byrv256zx1_3t79kv_fj880000gn/T/ipykernel_57880/2856192465.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sia["Date"] = pd.to_datetime(sia["Date"])


,Date,Value
0,2026-05-31,46.42M
1,2026-04-30,41.53M
2,2026-03-31,40.60M
3,2026-02-28,36.20M
4,2026-01-31,24.63M


In [46]:
# Merging:
merged = pd.merge(
    nvidia,
    sia[["Date", "Value"]],
    on="Date",
    how="left" # keep the all dates form nvidia set.
)

merged["Value"] = merged["Value"].fillna("")

## Output:
display(merged[merged["Value"] != ""].head(5)) # value column filled only at end of month.
display(merged.head(10))

,Date,Close,High,Low,Open,Volume,Value
607,2022-05-31,18.616369,19.142796,18.295328,18.923450,664100000,12.61M
628,2022-06-30,15.117031,15.523901,14.820853,15.318472,686070000,12.10M
671,2022-08-31,15.052211,15.496976,14.917584,15.341408,573710000,11.52M
692,2022-09-30,12.108989,12.601768,12.045147,12.057117,565638000,13.77M
713,2022-10-31,13.463634,13.803791,13.264128,13.743939,486341000,11.69M


,Date,Close,High,Low,Open,Volume,Value
0,2020-01-02,5.963805,5.963805,5.884506,5.934969,237536000,
1,2020-01-03,5.868347,5.912098,5.819375,5.844234,205384000,
2,2020-01-06,5.892957,5.898177,5.749026,5.775128,262636000,
3,2020-01-07,5.964302,6.010041,5.876302,5.921296,314856000,
4,2020-01-08,5.975488,6.016753,5.920053,5.960075,277108000,
5,2020-01-09,6.041113,6.113452,5.987419,6.061746,255112000,
6,2020-01-10,6.073429,6.178580,6.059259,6.148253,316296000,
7,2020-01-13,6.263846,6.288953,6.133836,6.156458,319840000,
8,2020-01-14,6.147011,6.246445,6.133835,6.221089,359088000,
9,2020-01-15,6.104502,6.182060,6.078649,6.159688,263104000,


### 2. Data Inspection

In [47]:
# Inspecting and cleaning time series data

## Date as index in both datasets:
sia.set_index("Date", inplace=True)
nvidia.set_index("Date", inplace=True)


In [48]:
## Duplicates
print(sia.duplicated().sum()) # 1 potential duplicate in sia.
print(nvidia.duplicated().sum())

1
0


In [49]:
sia[sia["Value"].duplicated(keep=False)] # duplicate is just the value in different months; duplicate should NOT be removed.

,Value
Date,
2025-03-31,20.71M
2024-11-30,20.71M


In [50]:
## Null
nvidia.isnull().sum()
sia.isnull().sum()
# NONE

Value    0
dtype: int64

In [ ]:
## Differences in time stamps (now, index)
nvidia.index.to_series().diff().value_counts()

Date
1 days    1276
3 days     291
4 days      48
2 days      15
Name: count, dtype: int64

In [70]:
## month-end data from nividia:
nvidia_monthly = nvidia.resample("ME").last()
nvidia_monthly.head(5)

,Close,High,Low,Open,Volume
Date,,,,,
2020-01-31,5.877297,6.076663,5.835535,6.064730,370420000
2020-02-29,6.717552,6.776999,6.014133,6.030798,1133252000
2020-03-31,6.556621,6.850127,6.411112,6.646165,949960000
2020-04-30,7.269989,7.423706,7.256060,7.369731,375916000
2020-05-31,8.830544,8.830544,8.442021,8.511169,745256000


### 2a. Feature Engineering & Merging Multi-Source Data